In [1]:
# ============================================================
# RECOMMENDER SYSTEMS — COMPLETE ML ENGINEER NOTEBOOK
# ============================================================
#
# ROADMAP
#
# 1. What is a Recommendation System?
# 2. Collaborative Filtering
# 3. Content-Based Filtering
# 4. Matrix Factorization
# 5. Implicit Feedback
# 6. Recommendation Metrics
# 7. Two-Tower Models
# 8. Retrieval + Ranking
# 9. Complete Production Architecture
#
# Goal:
# Understand how recommendation systems evolve from:
#
#   User-Item Matrix
#          ↓
#   Similarity / Latent Factors
#          ↓
#      Embeddings
#          ↓
#    Candidate Retrieval
#          ↓
#       Ranking
#          ↓
#   Final Recommendations
#
# IMPORTANT:
# For your ML Engineer + Applied AI/GenAI roadmap:
#
#   Collaborative Filtering      → understand
#   Content Based                → understand
#   Matrix Factorization         → understand
#   Implicit Feedback            → understand
#   Metrics                      → understand
#   Two-Tower                    → understand deeply
#   Retrieval + Ranking          → understand deeply
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt


# ============================================================
# 1. WHAT IS A RECOMMENDATION SYSTEM?
# ============================================================
#
# A recommendation system tries to answer:
#
#     "What should we show this user next?"
#
# Examples:
#
# Netflix  → movies
# Amazon   → products
# Spotify  → songs
# YouTube  → videos
# LinkedIn → jobs
#
#
# Basic architecture:
#
#                 USER
#                  ↓
#            User Features
#                  ↓
#             User Vector
#                  ↓
#          Candidate Retrieval
#                  ↓
#             1000 candidates
#                  ↓
#               Ranking
#                  ↓
#              Top 10
#                  ↓
#          Recommendation
#
#
# There are two major stages in modern systems:
#
# 1. RETRIEVAL
#       Find potentially relevant items.
#
# 2. RANKING
#       Order those items by predicted relevance.
#
#
# Why not rank every item?
#
# Suppose:
#
#     Users = 10 million
#     Items = 100 million
#
# Ranking all 100 million items for every request is expensive.
#
# Instead:
#
#     100M items
#          ↓
#     Retrieval
#          ↓
#     1000 candidates
#          ↓
#     Ranking
#          ↓
#     Top 10
#
# This retrieval → ranking architecture is extremely important
# for ML Engineer / Applied AI roles.


# ============================================================
# 2. TYPES OF FEEDBACK
# ============================================================
#
# Two major types:
#
# ------------------------------------------------------------
# EXPLICIT FEEDBACK
# ------------------------------------------------------------
#
# User explicitly tells us what they think.
#
# Example:
#
#     Movie rating = 5
#     Product rating = 1
#
# Matrix:
#
#             Movie A   Movie B   Movie C
# User 1         5         1         4
# User 2         4         0         5
#
#
# ------------------------------------------------------------
# IMPLICIT FEEDBACK
# ------------------------------------------------------------
#
# User doesn't explicitly rate something.
#
# We infer preference from behavior:
#
#     click
#     view
#     watch
#     purchase
#     add-to-cart
#     like
#
# Example:
#
#     User clicked product → positive signal
#     User purchased      → strong positive signal
#     User ignored       → unknown
#
# Important:
#
#     NOT CLICKED != DISLIKED
#
# This distinction becomes extremely important later.


# ============================================================
# 3. TOY DATASET
# ============================================================

ratings = pd.DataFrame(
    {
        "Movie_A": [5, 4, 0, 1, 0],
        "Movie_B": [4, 5, 0, 2, 1],
        "Movie_C": [0, 0, 5, 4, 5],
        "Movie_D": [1, 2, 4, 5, 4],
        "Movie_E": [0, 1, 5, 4, 5],
    },
    index=["User_1", "User_2", "User_3", "User_4", "User_5"]
)

print(ratings)


# ============================================================
# 4. COLLABORATIVE FILTERING
# ============================================================
#
# IDEA:
#
#     Users who behaved similarly in the past
#     may like similar items in the future.
#
#
# Example:
#
# User A:
#
#     Movie A = 5
#     Movie B = 4
#     Movie D = 1
#
# User B:
#
#     Movie A = 4
#     Movie B = 5
#     Movie D = 2
#
# Their behavior is similar.
#
# Therefore:
#
#     If User A likes a movie that User B has watched,
#     recommend it to User A.
#
#
# Two common approaches:
#
#     1. User-based CF
#     2. Item-based CF
#
#
# ============================================================
# 4.1 USER-BASED COLLABORATIVE FILTERING
# ============================================================

user_similarity = cosine_similarity(ratings)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=ratings.index,
    columns=ratings.index
)

print(user_similarity_df)


# ------------------------------------------------------------
# Find users similar to User_1
# ------------------------------------------------------------

target_user = "User_1"

similar_users = (
    user_similarity_df[target_user]
    .sort_values(ascending=False)
)

print(similar_users)


# ============================================================
# USER-BASED WORKING
# ============================================================
#
# User_1
#   ↓
# Compare with all users
#   ↓
# Calculate similarity
#   ↓
# Find similar users
#   ↓
# Look at items they liked
#   ↓
# Recommend unseen items
#
#
# Main problem:
#
# User-user similarity becomes expensive when the number
# of users becomes huge.
#
# This is one reason item-based methods and embedding-based
# retrieval became important.


# ============================================================
# 4.2 ITEM-BASED COLLABORATIVE FILTERING
# ============================================================
#
# Instead of:
#
#     "Which users are similar?"
#
# ask:
#
#     "Which items are similar?"
#
#
# Transpose matrix:
#
#     Users × Items
#
# becomes:
#
#     Items × Users
#
# ------------------------------------------------------------

item_similarity = cosine_similarity(ratings.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=ratings.columns,
    columns=ratings.columns
)

print(item_similarity_df)


# ------------------------------------------------------------
# Similar items to Movie_A
# ------------------------------------------------------------

print(
    item_similarity_df["Movie_A"]
    .sort_values(ascending=False)
)


# ============================================================
# ITEM-BASED WORKING
# ============================================================
#
# User liked Movie_A
#       ↓
# Find items similar to Movie_A
#       ↓
# Movie_B / Movie_D / ...
#       ↓
# Recommend similar items
#
#
# This idea is conceptually similar to:
#
#     embedding
#         ↓
#     vector similarity
#
# which we will reach later.


# ============================================================
# 5. CONTENT-BASED FILTERING
# ============================================================
#
# Collaborative Filtering:
#
#     "Users who liked similar things..."
#
#
# Content-Based:
#
#     "This item has similar CONTENT to what you liked."
#
#
# Example:
#
# User liked:
#
#     "action science fiction space"
#
# Recommend:
#
#     "science fiction space adventure"
#
#
# We need item features.
#
# Example:
#
# Movie A → action sci-fi space
# Movie B → comedy romance
# Movie C → sci-fi space adventure
#
#
# Convert text/features into vectors.
#
# Then:
#
#     User profile vector
#             ↓
#     cosine similarity
#             ↓
#     similar items


movies = pd.DataFrame(
    {
        "movie": [
            "Movie_A",
            "Movie_B",
            "Movie_C",
            "Movie_D",
            "Movie_E"
        ],
        "description": [
            "action science fiction space",
            "romance comedy relationship",
            "science fiction space adventure",
            "action thriller crime",
            "science fiction adventure future"
        ]
    }
)

print(movies)


# ============================================================
# 5.1 TF-IDF REPRESENTATION
# ============================================================

vectorizer = TfidfVectorizer()

movie_vectors = vectorizer.fit_transform(
    movies["description"]
)

print(movie_vectors.shape)


# ============================================================
# 5.2 ITEM-ITEM SIMILARITY
# ============================================================

content_similarity = cosine_similarity(movie_vectors)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=movies["movie"],
    columns=movies["movie"]
)

print(content_similarity_df)


# ============================================================
# 5.3 RECOMMEND SIMILAR ITEMS
# ============================================================

target_movie = "Movie_A"

recommendations = (
    content_similarity_df[target_movie]
    .sort_values(ascending=False)
)

print(recommendations)


# ============================================================
# CONTENT-BASED WORKING
# ============================================================
#
# Item
#  ↓
# Features
#  ↓
# Vector representation
#  ↓
# Similarity
#  ↓
# Similar items
#  ↓
# Recommendation
#
#
# Modern systems replace simple TF-IDF features with:
#
#     Neural embeddings
#
# Example:
#
# text → embedding model → 768-dimensional vector
#
# Then:
#
#     cosine similarity / dot product
#
# This is directly connected to semantic search and RAG.


# ============================================================
# 6. MATRIX FACTORIZATION
# ============================================================
#
# Collaborative filtering can be represented using:
#
#             USER × ITEM
#
# Example:
#
#                 A   B   C   D   E
# User_1         5   4   0   1   0
# User_2         4   5   0   2   1
#
#
# Matrix factorization tries to represent this large matrix
# using smaller latent matrices.
#
#
#     R ≈ U × V^T
#
#
# Where:
#
#     R = User-Item rating matrix
#
#     U = User latent vectors
#
#     V = Item latent vectors
#
#
# Example:
#
#     User_1 → [0.8, 0.2]
#     User_2 → [0.7, 0.3]
#
#     Movie_A → [0.9, 0.1]
#     Movie_B → [0.8, 0.2]
#
#
# These dimensions may represent hidden concepts such as:
#
#     action preference
#     comedy preference
#     popularity preference
#
# They are NOT necessarily human-interpretable.


# ============================================================
# 6.1 SIMPLE MATRIX FACTORIZATION IDEA
# ============================================================
#
# Predicted rating:
#
#     r_hat(u,i) = p_u · q_i
#
#
# where:
#
#     p_u = user vector
#     q_i = item vector
#
#
# Expanded:
#
#     r_hat = p1*q1 + p2*q2 + ... + pk*qk
#
#
# Training tries to minimize:
#
#     Σ (r_ui - p_u · q_i)^2
#
#
# plus regularization:
#
#     + λ(||p_u||² + ||q_i||²)
#
#
# Why regularization?
#
# Prevents latent vectors from becoming too large
# and reduces overfitting.


# ============================================================
# 6.2 SIMPLE MATRIX FACTORIZATION IMPLEMENTATION
# ============================================================
#
# This is an educational implementation.
#
# It is intentionally simple so you can understand the
# internal working.
#
# ============================================================

R = ratings.values.astype(float)

num_users, num_items = R.shape

k = 2

np.random.seed(42)

U = np.random.normal(
    0,
    0.1,
    (num_users, k)
)

V = np.random.normal(
    0,
    0.1,
    (num_items, k)
)

learning_rate = 0.01
regularization = 0.01

epochs = 500


# Train only on observed ratings
observed = np.argwhere(R > 0)

for epoch in range(epochs):

    for u, i in observed:

        # Predicted rating
        prediction = np.dot(U[u], V[i])

        # Error
        error = R[u, i] - prediction

        # Save old vectors because both are updated
        user_vector = U[u].copy()
        item_vector = V[i].copy()

        # Gradient update
        U[u] += learning_rate * (
            error * item_vector
            - regularization * user_vector
        )

        V[i] += learning_rate * (
            error * user_vector
            - regularization * item_vector
        )


# Reconstruct rating matrix
predicted_ratings = U @ V.T

predicted_df = pd.DataFrame(
    predicted_ratings,
    index=ratings.index,
    columns=ratings.columns
)

print(predicted_df)


# ============================================================
# MATRIX FACTORIZATION WORKING
# ============================================================
#
# Rating Matrix
#       ↓
# Initialize User Vectors
#       ↓
# Initialize Item Vectors
#       ↓
# Predict rating
#       ↓
# Calculate error
#       ↓
# Update user/item vectors
#       ↓
# Repeat
#       ↓
# Learned embeddings
#
#
# IMPORTANT CONNECTION:
#
# Matrix Factorization
#          ↓
# User embedding + Item embedding
#          ↓
# Similarity / dot product
#
# This is the conceptual foundation for modern
# two-tower retrieval models.


# ============================================================
# 7. IMPLICIT FEEDBACK
# ============================================================
#
# Real recommendation systems often don't have ratings.
#
# Instead:
#
#     user clicked product
#     user watched video
#     user searched item
#     user purchased item
#
#
# Example:
#
#             Item_A Item_B Item_C Item_D
# User_1        1      1      0      0
# User_2        0      1      1      0
#
#
# Here:
#
#     1 = interaction
#     0 = no observed interaction
#
#
# IMPORTANT:
#
#     0 DOES NOT necessarily mean dislike.
#
# It may mean:
#
#     "We don't know."


implicit = pd.DataFrame(
    {
        "Item_A": [1, 0, 1, 0],
        "Item_B": [1, 1, 1, 0],
        "Item_C": [0, 1, 1, 1],
        "Item_D": [0, 0, 1, 1],
    },
    index=["User_1", "User_2", "User_3", "User_4"]
)

print(implicit)


# ============================================================
# 7.1 DIFFERENT INTERACTION STRENGTHS
# ============================================================
#
# Not all interactions are equally strong.
#
# Example:
#
#     impression       → weak
#     click             → stronger
#     add-to-cart       → stronger
#     purchase          → very strong
#
#
# We can create weighted signals:
#
#     impression = 1
#     click      = 3
#     cart       = 5
#     purchase   = 10
#
#
# This is called confidence / interaction weighting.
#
# Real systems often use more sophisticated approaches.


events = {
    "impression": 1,
    "click": 3,
    "add_to_cart": 5,
    "purchase": 10
}

print(events)


# ============================================================
# 7.2 POSITIVE / NEGATIVE / UNKNOWN
# ============================================================
#
# A useful mental model:
#
#     Positive:
#         clicked / purchased / liked
#
#     Unknown:
#         never interacted
#
#     Negative:
#         explicit dislike / skip / negative feedback
#
#
# Don't automatically treat every missing interaction
# as a negative label.


# ============================================================
# 8. RECOMMENDATION METRICS
# ============================================================
#
# Recommendation is usually a ranking problem.
#
# Therefore accuracy alone is often not enough.
#
# We care about:
#
#     Precision@K
#     Recall@K
#     MAP@K
#     NDCG@K
#
#
# ============================================================
# 8.1 PRECISION@K
# ============================================================
#
# Precision@K =
#
# relevant recommended items
# --------------------------
#         K
#
#
# Example:
#
# Recommended:
#
#     [A, B, C, D, E]
#
# Relevant:
#
#     [A, C]
#
# K = 5
#
# Precision@5 = 2/5 = 0.4


def precision_at_k(recommended, relevant, k):

    recommended = recommended[:k]

    hits = sum(
        item in relevant
        for item in recommended
    )

    return hits / k


recommended = ["A", "B", "C", "D", "E"]
relevant = {"A", "C"}

print(
    "Precision@5:",
    precision_at_k(recommended, relevant, 5)
)


# ============================================================
# 8.2 RECALL@K
# ============================================================
#
# Recall@K =
#
# relevant recommended items
# --------------------------
# total relevant items
#
#
# Example:
#
# Total relevant = 4
# Retrieved relevant = 2
#
# Recall = 2/4 = 0.5


def recall_at_k(recommended, relevant, k):

    recommended = recommended[:k]

    hits = sum(
        item in relevant
        for item in recommended
    )

    if len(relevant) == 0:
        return 0

    return hits / len(relevant)


print(
    "Recall@5:",
    recall_at_k(recommended, relevant, 5)
)


# ============================================================
# 8.3 AVERAGE PRECISION / MAP@K
# ============================================================
#
# MAP = Mean Average Precision
#
# It cares about:
#
#     How early do relevant items appear?
#
#
# If relevant items appear near the top,
# the score is higher.


def average_precision_at_k(recommended, relevant, k):

    hits = 0
    precision_sum = 0.0

    for i, item in enumerate(recommended[:k], start=1):

        if item in relevant:

            hits += 1

            precision = hits / i

            precision_sum += precision

    if len(relevant) == 0:
        return 0

    return precision_sum / min(len(relevant), k)


print(
    "AP@5:",
    average_precision_at_k(
        recommended,
        relevant,
        5
    )
)


# ============================================================
# 8.4 NDCG@K
# ============================================================
#
# NDCG = Normalized Discounted Cumulative Gain
#
# Main idea:
#
#     A relevant item at rank 1 is more valuable
#     than the same item at rank 10.
#
#
# DCG:
#
#     relevance_i
#     ------------
#     log2(i + 1)
#
#
# NDCG:
#
#     DCG
#     ---
#     IDCG
#
# Range:
#
#     0 → 1
#
# Higher means relevant items are ranked earlier.
#
#
# This is especially useful for ranking systems.


def ndcg_at_k(relevances, k):

    relevances = np.asarray(
        relevances[:k],
        dtype=float
    )

    if len(relevances) == 0:
        return 0

    discounts = np.log2(
        np.arange(2, len(relevances) + 2)
    )

    dcg = np.sum(
        relevances / discounts
    )

    ideal = np.sort(
        relevances
    )[::-1]

    idcg = np.sum(
        ideal / discounts
    )

    if idcg == 0:
        return 0

    return dcg / idcg


# Example:
#
# 1 = relevant
# 0 = not relevant
#
relevance = [1, 0, 1, 0, 1]

print(
    "NDCG@5:",
    ndcg_at_k(relevance, 5)
)


# ============================================================
# METRICS SUMMARY
# ============================================================
#
# Precision@K
#     "How many recommended items were relevant?"
#
# Recall@K
#     "How many of all relevant items did we retrieve?"
#
# MAP@K
#     "How good is the ranking across relevant items?"
#
# NDCG@K
#     "Are relevant items appearing near the top?"
#
#
# Interview:
#
# Precision → quality of retrieved recommendations
# Recall    → coverage of relevant items
# NDCG      → ranking quality with position importance
#
#
# ============================================================


# ============================================================
# 9. TWO-TOWER MODEL
# ============================================================
#
# NOW WE MOVE TO MODERN RECOMMENDATION SYSTEMS.
#
#
# Instead of comparing raw users/items:
#
#     User → vector
#     Item → vector
#
# Then:
#
#     similarity(user_vector, item_vector)
#
#
# Two towers:
#
#
#             USER                    ITEM
#              │                       │
#       User features            Item features
#              │                       │
#        User Tower              Item Tower
#              │                       │
#        User Embedding          Item Embedding
#              │                       │
#              └─────────┬─────────────┘
#                        ↓
#                   Dot Product
#                        ↓
#                     Score
#
#
# Why two towers?
#
# Item embeddings can be computed ahead of time.
#
# Then store item vectors in a vector index.
#
# At request time:
#
#     user
#      ↓
# user embedding
#      ↓
# vector search
#      ↓
# top-K items
#
#
# This is extremely important for large-scale retrieval.


# ============================================================
# 9.1 SIMPLE TWO-TOWER IDEA
# ============================================================
#
# We won't train a deep neural network here yet.
#
# First understand the architecture.
#
# User vector:
#
#     u = [0.8, 0.2, 0.5]
#
# Item vector:
#
#     v = [0.7, 0.3, 0.6]
#
#
# Score:
#
#     u · v
#
#
# Higher score → more similar/preferred.


user_embedding = np.array([0.8, 0.2, 0.5])

item_embeddings = {
    "Item_A": np.array([0.7, 0.3, 0.6]),
    "Item_B": np.array([0.1, 0.9, 0.2]),
    "Item_C": np.array([0.8, 0.2, 0.4]),
    "Item_D": np.array([0.2, 0.1, 0.9]),
}


scores = {}

for item, embedding in item_embeddings.items():

    scores[item] = np.dot(
        user_embedding,
        embedding
    )


print(scores)


# Sort recommendations
sorted_items = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(sorted_items)


# ============================================================
# TWO-TOWER WORKING
# ============================================================
#
# User
#   ↓
# User Features
#   ↓
# User Neural Network
#   ↓
# User Embedding
#          \
#           \
#            → Dot Product → Similarity
#           /
#          /
# Item Embedding
#   ↑
# Item Neural Network
#   ↑
# Item Features
#
#
# Training:
#
#     Positive user-item pairs
#     Negative user-item pairs
#
# Objective:
#
#     Positive pair → high similarity
#     Negative pair → low similarity
#
#
# Common losses:
#
#     Contrastive loss
#     Triplet loss
#     Softmax / sampled softmax
#     InfoNCE-style objectives
#
#
# You do NOT need to implement all of these from scratch
# for your current roadmap.
#
# Understand the architecture and training idea deeply.


# ============================================================
# 10. VECTOR SEARCH
# ============================================================
#
# Suppose we have:
#
#     10 million item embeddings
#
# For every user:
#
#     calculate similarity against all 10 million
#
# This can be expensive.
#
#
# Instead use an Approximate Nearest Neighbor (ANN) index.
#
#
# Examples:
#
#     FAISS
#     HNSW
#     ScaNN
#     vector databases
#
#
# Concept:
#
# User embedding
#       ↓
# Vector index
#       ↓
# nearest neighbors
#       ↓
# candidate items
#
#
# For your Applied AI/GenAI path, this concept is VERY important
# because RAG uses almost exactly the same retrieval idea.


# ============================================================
# 10.1 SIMPLE VECTOR SEARCH WITHOUT A VECTOR DATABASE
# ============================================================
#
# We can demonstrate the idea with cosine similarity.
#
# ------------------------------------------------------------

item_names = list(item_embeddings.keys())

item_matrix = np.vstack(
    [item_embeddings[item] for item in item_names]
)

similarities = cosine_similarity(
    user_embedding.reshape(1, -1),
    item_matrix
)[0]

retrieved = sorted(
    zip(item_names, similarities),
    key=lambda x: x[1],
    reverse=True
)

print("Retrieved candidates:")
print(retrieved)


# ============================================================
# VECTOR SEARCH WORKING
# ============================================================
#
# User
#  ↓
# User embedding
#  ↓
# Query vector
#  ↓
# ANN / Vector Search
#  ↓
# Top-K nearest vectors
#  ↓
# Candidate set
#
#
# IMPORTANT:
#
# Vector search itself is usually NOT the final recommendation.
#
# It generates candidates.
#
# Then ranking happens.


# ============================================================
# 11. RETRIEVAL + RANKING
# ============================================================
#
# This is one of the most important concepts for you.
#
#
# LARGE ITEM SET
#
# 10,000,000 items
#        ↓
#      RETRIEVAL
#        ↓
#      1,000 items
#        ↓
#       RANKING
#        ↓
#       100 items
#        ↓
#    BUSINESS FILTERS
#        ↓
#        10 items
#
#
# Why separate retrieval and ranking?
#
# Retrieval needs:
#
#     VERY FAST
#     high recall
#     scalable
#
#
# Ranking can be:
#
#     MORE COMPLEX
#     more features
#     more accurate
#
#
# Because it only processes 1000 candidates instead of
# 10 million items.


# ============================================================
# 11.1 RETRIEVAL FEATURES
# ============================================================
#
# Retrieval might use:
#
#     user embedding
#     item embedding
#     similarity
#
#
# Ranking can additionally use:
#
#     user age
#     location
#     item popularity
#     price
#     previous interactions
#     freshness
#     time of day
#     device
#     category
#
#
# Example:
#
# Candidate 1:
#     similarity = 0.91
#     popularity = 0.8
#     freshness = 0.9
#
# Candidate 2:
#     similarity = 0.88
#     popularity = 0.4
#     freshness = 0.95
#
# A ranking model combines these signals.


# ============================================================
# 11.2 SIMPLE RANKING MODEL
# ============================================================
#
# For learning purposes, let's create candidate features.
#
# In real systems this could be:
#
#     Logistic Regression
#     Gradient Boosting
#     XGBoost
#     neural ranking model
#
#
# Example score:
#
#     score =
#         0.6 * similarity
#       + 0.2 * popularity
#       + 0.2 * freshness


candidates = pd.DataFrame(
    {
        "item": ["A", "B", "C", "D", "E"],
        "similarity": [0.91, 0.88, 0.82, 0.79, 0.75],
        "popularity": [0.5, 0.9, 0.4, 0.8, 0.3],
        "freshness": [0.9, 0.3, 0.8, 0.7, 0.95]
    }
)

candidates["ranking_score"] = (
    0.6 * candidates["similarity"]
    + 0.2 * candidates["popularity"]
    + 0.2 * candidates["freshness"]
)

ranked = candidates.sort_values(
    "ranking_score",
    ascending=False
)

print(ranked)


# ============================================================
# RETRIEVAL + RANKING WORKING TREE
# ============================================================
#
#                  ALL ITEMS
#                10,000,000
#                      │
#                      ↓
#                RETRIEVAL
#                      │
#                      ↓
#                   1,000
#                      │
#                      ↓
#                   RANKER
#                      │
#                      ↓
#                    100
#                      │
#                      ↓
#             Business Filtering
#                      │
#                      ↓
#                     10
#                      │
#                      ↓
#              FINAL RECOMMENDATION
#
#
# THIS ARCHITECTURE IS VERY IMPORTANT.
#
# It appears in:
#
#     Recommendation
#     Search
#     Semantic Search
#     RAG
#     Ads
#     Feed Ranking
#     Job Search
#     Product Search


# ============================================================
# 12. RETRIEVAL VS RANKING
# ============================================================
#
# RETRIEVAL
#
# Goal:
#     Don't miss relevant candidates.
#
# Main metric:
#     Recall@K
#
# Characteristics:
#
#     fast
#     scalable
#     approximate
#     vector search
#
#
# RANKING
#
# Goal:
#     Put the best candidates at the top.
#
# Main metrics:
#
#     NDCG
#     MAP
#     Precision@K
#
# Characteristics:
#
#     more expensive
#     more features
#     sophisticated models
#
#
# Interview question:
#
# "Why not use one huge ranking model?"
#
# Answer:
#
# Because scoring every item for every user is computationally
# expensive. Retrieval reduces the search space first, allowing
# a more sophisticated ranking model to operate on a small
# candidate set.


# ============================================================
# 13. COMPLETE MODERN RECOMMENDATION ARCHITECTURE
# ============================================================
#
#
#                         USER
#                           │
#                           ↓
#                    User Features
#                           │
#                           ↓
#                     User Tower
#                           │
#                           ↓
#                    User Embedding
#                           │
#                           ↓
#                  Vector Retrieval
#                           │
#                           ↓
#                    1000 Candidates
#                           │
#                           ↓
#                    Ranking Model
#                           │
#                           ↓
#                     100 Candidates
#                           │
#                           ↓
#                Business / Safety Rules
#                           │
#                           ↓
#                       Top 10
#                           │
#                           ↓
#                    RECOMMENDATION
#
#
# Item side:
#
#     Item metadata
#          ↓
#     Item Tower
#          ↓
#     Item Embedding
#          ↓
#     Vector Index
#
#
# Item embeddings can often be precomputed and indexed.
#
#
# ============================================================


# ============================================================
# 14. COMPLETE END-TO-END MENTAL MODEL
# ============================================================
#
#
#               CLASSICAL RECOMMENDATION
#
# User-Item Matrix
#       ↓
# Collaborative Filtering
#       ↓
# Similarity
#
#
#               LATENT REPRESENTATION
#
# User-Item Matrix
#       ↓
# Matrix Factorization
#       ↓
# User Embeddings + Item Embeddings
#
#
#               MODERN RECOMMENDATION
#
# User Features ──→ User Tower ──→ User Embedding
#                                      │
#                                      ↓
#                                Vector Search
#                                      ↑
#                                      │
# Item Features ──→ Item Tower ──→ Item Embedding
#
#                                      ↓
#                               Candidates
#                                      ↓
#                                   Ranker
#                                      ↓
#                                Top Results
#
#
# ============================================================
# 15. CONNECTION TO RAG / GENAI
# ============================================================
#
# This is VERY important for your career path.
#
#
# RECOMMENDER:
#
# User
#   ↓
# User embedding
#   ↓
# Vector search
#   ↓
# Candidate items
#   ↓
# Ranking
#
#
# RAG:
#
# User Query
#   ↓
# Query embedding
#   ↓
# Vector search
#   ↓
# Candidate documents/chunks
#   ↓
# Reranking
#   ↓
# LLM
#   ↓
# Answer
#
#
# Therefore:
#
# Recommendation Systems
#          ↓
# Embeddings
#          ↓
# Vector Search
#          ↓
# Retrieval
#          ↓
# Ranking
#
# directly prepares you for:
#
#     Semantic Search
#     RAG
#     Vector Databases
#     Retrieval Systems
#     LLM applications
#
#
# ============================================================
# 16. INTERVIEW QUESTIONS
# ============================================================
#
# Q1. What is collaborative filtering?
#
# Answer:
# Collaborative filtering recommends items based on patterns
# in user-item interactions rather than item content.
#
#
# Q2. User-based vs item-based collaborative filtering?
#
# User-based:
#     find similar users.
#
# Item-based:
#     find similar items.
#
#
# Q3. What is matrix factorization?
#
# Answer:
# It approximates the user-item interaction matrix as the
# product of lower-dimensional user and item latent matrices.
#
#
#     R ≈ U V^T
#
#
# Q4. What is implicit feedback?
#
# Answer:
# Behavioral signals such as clicks, views, purchases and
# watch time that indirectly indicate user preference.
#
#
# Q5. Why is missing interaction not necessarily negative?
#
# Because the user may simply never have seen the item.
#
#
# Q6. What is a two-tower model?
#
# Answer:
# A model with separate networks that encode users and items
# into a shared embedding space. Their embeddings are compared
# using a similarity function such as dot product.
#
#
# Q7. Why use two towers?
#
# Item embeddings can be precomputed and indexed, making
# large-scale candidate retrieval efficient.
#
#
# Q8. Retrieval vs ranking?
#
# Retrieval:
#     find a relatively small candidate set.
#
# Ranking:
#     order those candidates using richer features/models.
#
#
# Q9. Why not rank all items?
#
# Computational cost.
#
#
# Q10. What is Precision@K?
#
# Relevant recommendations / K.
#
#
# Q11. What is Recall@K?
#
# Relevant recommended items / total relevant items.
#
#
# Q12. What is NDCG?
#
# A ranking metric that gives more importance to relevant
# results appearing near the top.
#
#
# Q13. How is recommendation related to RAG?
#
# Both can use:
#
#     embeddings
#     vector search
#     candidate retrieval
#     reranking
#
# RAG retrieves documents instead of products/items and then
# provides them to an LLM.


# ============================================================
# 17. 30-SECOND INTERVIEW EXPLANATION
# ============================================================
#
# "A modern recommendation system usually has two stages.
# First, a retrieval system generates a relatively small set
# of candidate items, often using embeddings and vector
# similarity. Then a ranking model scores those candidates
# using richer user, item and contextual features.
#
# Classical approaches include collaborative filtering and
# matrix factorization, while modern systems often use
# two-tower neural networks to learn user and item embeddings.
#
# The architecture is designed to balance recommendation
# quality with large-scale serving efficiency."


# ============================================================
# 18. FINAL CHEAT SHEET
# ============================================================
#
# COLLABORATIVE FILTERING
# -----------------------
# Input:
#     user-item interactions
#
# Idea:
#     similar users/items → similar preferences
#
#
# CONTENT-BASED
# -------------
# Input:
#     item features
#
# Idea:
#     recommend items similar to what user liked
#
#
# MATRIX FACTORIZATION
# --------------------
# Idea:
#
#     R ≈ U V^T
#
# Learns:
#
#     user latent vectors
#     item latent vectors
#
#
# IMPLICIT FEEDBACK
# -----------------
# Signals:
#
#     click
#     view
#     purchase
#     watch
#
# Important:
#
#     missing != negative
#
#
# METRICS
# -------
#
# Precision@K
#     recommendation quality
#
# Recall@K
#     retrieval coverage
#
# MAP@K
#     ranking quality across relevant items
#
# NDCG@K
#     ranking quality with position importance
#
#
# TWO-TOWER
# ---------
#
# User → User Tower → User Embedding
#
# Item → Item Tower → Item Embedding
#
#                 ↓
#             similarity
#
#
# RETRIEVAL
# ---------
#
# Millions of items
#       ↓
# vector search
#       ↓
# hundreds/thousands candidates
#
#
# RANKING
# -------
#
# candidates
#     ↓
# rich features
#     ↓
# ranking model
#     ↓
# top results
#
#
# ============================================================
# 19. FINAL WORKING TREE
# ============================================================
#
#
#                       RECOMMENDER SYSTEM
#                              │
#             ┌────────────────┴────────────────┐
#             │                                 │
#             ↓                                 ↓
#      Classical ML                     Modern ML / AI
#             │                                 │
#     ┌───────┴────────┐                ┌──────┴──────┐
#     ↓                ↓                ↓             ↓
# Collaborative    Content-Based    Two-Tower     Embeddings
# Filtering        Filtering        Models            │
#     │                │                │              ↓
#     └───────┬────────┘                └──────→ Vector Search
#             ↓                                  │
#     Matrix Factorization                       ↓
#             │                             Retrieval
#             ↓                                │
#      Latent Embeddings                        ↓
#                                           Candidates
#                                                │
#                                                ↓
#                                             Ranking
#                                                │
#                                                ↓
#                                          Top-K Results
#
#
# ============================================================
# 20. WHAT YOU SHOULD REMEMBER
# ============================================================
#
# The most important progression is:
#
#     User-Item Matrix
#            ↓
#     Collaborative Filtering
#            ↓
#     Matrix Factorization
#            ↓
#     User / Item Embeddings
#            ↓
#       Two-Tower Models
#            ↓
#      Vector Retrieval
#            ↓
#       Candidate Set
#            ↓
#          Ranking
#            ↓
#      Final Recommendation
#
#
# AND THIS IS THE BIG CONNECTION:
#
#     Recommendation
#          ↓
#     Embeddings
#          ↓
#     Vector Search
#          ↓
#     Retrieval
#          ↓
#     Ranking
#          ↓
#     RAG / Semantic Search / GenAI
#
# ============================================================

        Movie_A  Movie_B  Movie_C  Movie_D  Movie_E
User_1        5        4        0        1        0
User_2        4        5        0        2        1
User_3        0        0        5        4        5
User_4        1        2        4        5        4
User_5        0        1        5        4        5
          User_1    User_2    User_3    User_4    User_5
User_1  1.000000  0.955533  0.075974  0.352738  0.150809
User_2  0.955533  1.000000  0.235935  0.524304  0.324232
User_3  0.075974  0.235935  1.000000  0.937958  0.992509
User_4  0.352738  0.524304  0.937958  1.000000  0.961963
User_5  0.150809  0.324232  0.992509  0.961963  1.000000
User_1    1.000000
User_2    0.955533
User_4    0.352738
User_5    0.150809
User_3    0.075974
Name: User_1, dtype: float64
          Movie_A   Movie_B   Movie_C   Movie_D   Movie_E
Movie_A  1.000000  0.955533  0.075974  0.352738  0.150809
Movie_B  0.955533  1.000000  0.235935  0.524304  0.324232
Movie_C  0.075974  0.235935  1.000000  0.937958 